# 04 · Verification and Leakage Audit

Confirms KL-scale consistency across datasets and audits for subject-level leakage.
Any subject appearing in multiple splits within a dataset is reassigned to a single
split (its majority split), eliminating leakage. The leave-one-dataset-out protocol
ensures the external set is disjoint from training by construction.

## Setup

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np
manifest=pd.read_csv(str(config.MANIFEST_CSV))
print(f'Manifest: {len(manifest):,} rows')

Mounted at /content/drive
Manifest: 61,591 rows


## Label-scale consistency

In [5]:
ok=True

if 'dataset' not in manifest.columns:
    manifest['dataset'] = manifest['filename'].apply(lambda x: x.split('_')[0].lower())

for name in config.DATASETS:
    sub=manifest[manifest['dataset']==name]
    if len(sub)==0: print(f'  {name}: none'); ok=False; continue
    vals=sorted(sub['kl_grade'].unique().tolist()); inr=set(vals).issubset({0,1,2,3,4})
    print(f'  {name:10s}: grades {vals}  {"ok" if inr else "OUT OF RANGE"}')
    ok=ok and inr
print('Consistent.' if ok else 'Fix scales.')

  oai       : grades [0, 1, 2, 3, 4]  ok
  nhanes3   : grades [0, 1, 2, 3, 4]  ok
  mrkr      : grades [0, 1, 2, 3, 4]  ok
  mendeley  : grades [0, 1, 2, 3, 4]  ok
Consistent.


## Subject-level leakage audit and correction

Each subject's images are assigned to their majority split, so no subject spans multiple splits within a dataset.

In [9]:

subj_col = None
for cand in ['subject_id', 'subject', 'patient_id', 'patient', 'id', 'empi_anon']:
    if cand in manifest.columns:
        subj_col = cand
        break

if subj_col is None:
    import re
    def extract_subject(fn):
        stem = str(fn).rsplit('.', 1)[0]
        token = stem.split('_')[-1]
        token = re.sub(r'[LR]$', '', token)
        return token if token else stem
    manifest['subject_id'] = manifest['filename'].apply(extract_subject)
    subj_col = 'subject_id'
    print(f"Derived subject_id from filename (e.g. {manifest['subject_id'].iloc[0]})")
else:
    print(f"Using existing subject column: '{subj_col}'")

if 'split' not in manifest.columns:
    manifest['split'] = 'train'
    print("No split column found; defaulted all to 'train'")

majority = (manifest.groupby(['dataset', subj_col])['split']
            .agg(lambda s: s.mode()[0]).rename('split_fixed'))
manifest = manifest.merge(majority, on=['dataset', subj_col], how='left')
manifest['split'] = manifest['split_fixed']
manifest = manifest.drop(columns=['split_fixed'])
manifest.to_csv(str(config.MANIFEST_CSV), index=False)

leak = False
for name in config.DATASETS:
    sub = manifest[manifest['dataset'] == name]
    if len(sub) == 0:
        continue
    n = int((sub.groupby(subj_col)['split'].nunique() > 1).sum())
    print(f'  {name:10s}: {n} subjects in multiple splits {"<-LEAK" if n else "(clean)"}')
    leak = leak or n > 0
print('PASSED - manifest corrected and saved.' if not leak else 'LEAK REMAINS')

Derived subject_id from filename (e.g. 224px)
  oai       : 0 subjects in multiple splits (clean)
  nhanes3   : 0 subjects in multiple splits (clean)
  mrkr      : 0 subjects in multiple splits (clean)
  mendeley  : 0 subjects in multiple splits (clean)
PASSED - manifest corrected and saved.


In [10]:
print(manifest[['filename', 'subject_id', 'dataset']].head(10))

                            filename subject_id dataset
0  OAI_9000099_L_clahe-2.0_224px.png      224px     oai
1  OAI_9000099_R_clahe-2.0_224px.png      224px     oai
2  OAI_9000296_L_clahe-2.0_224px.png      224px     oai
3  OAI_9000296_R_clahe-2.0_224px.png      224px     oai
4  OAI_9000622_L_clahe-2.0_224px.png      224px     oai
5  OAI_9000622_R_clahe-2.0_224px.png      224px     oai
6  OAI_9000798_L_clahe-2.0_224px.png      224px     oai
7  OAI_9000798_R_clahe-2.0_224px.png      224px     oai
8  OAI_9001104_L_clahe-2.0_224px.png      224px     oai
9  OAI_9001104_R_clahe-2.0_224px.png      224px     oai
